# Attendance Grading

* Step 1: Download an updated version of the gradebook
* Step 2: Input the proper session length into the session_length variable
* Step 3: Input the csv names into the respective variables
* Step 4: Input the assignment name from the gradebook into the assignment_name variable
* Step 5: Make sure the zoom recording csv has records occuring after the due date removed before running results.

Results are retrieved in "grading_results.csv". The remaining csv's are used for debugging purposes. "Testing.csv" returns with more columns joined to the gradebook: "Name", "Duration (minutes)" "View Duration (minutes)", "Total View Time (minutes) and "Grade" to verify that the grading script is placing the correct grades in the correct student's row. "Testing.csv" is for debugging/testing purposes not to be put on Canvas.

## CSV's used for testing/debugging purposes
* Preliminary.csv
* Testing.csv
* grade_count.csv
* live_and_recording.csv
* grade_verification.csv

## CSV's required for script to function
* Gradebook csv (gradebook_name)
* Zoom recording csv (recording_csv_name)
* Zoom live csv (live_csv_name)

## Other necessary inputs
* Zoom session length (session_length)
* Name of assignment in gradebook (assignment_name)

Final results are placed in the "grading_results.csv" use that for inputting into canvas.

TODO: Removing columns from the zoom recording csv where attendance is inadmissible for credit (student watched too late) automatically rather than having to manually remove them.

In [9]:
import pandas as pd

# Set the variable session_length to how long the zoom session was for to accurately get grading results (in minutes)
session_length = 170


# Name of attendance csv (change as needed)
recording_csv_name = "zoomus_recording_02-02-2026..csv"
live_csv_name = "zoomus_live_02-02-2026.csv"

# Name of assignment in the gradebook (change as needed)
assignment_name = "2/2/26 Attendance (2598434)"

# Gradebook name (change as needed)
gradebook_name = "2026-02-11T0033_Grades-COP4808_001_13815.csv" 

# Date where attendance viewing is no longer counted. Format in mm-dd-yyyy

In [10]:
# Read initial zoom csv recording and live
live_df = pd.read_csv(live_csv_name)
recording_df = pd.read_csv(recording_csv_name, index_col=False)
recording_df = recording_df.rename(columns={"User Email": "Email"})

print(recording_df.head())
# Fill in 0's for "< 1" minute in recording and live csv's 
live_df['Total duration (minutes)'] = live_df['Total duration (minutes)'].apply(lambda x: 0 if x == '< 1' else int(x))
# recording_df["View Duration (minutes)"] = recording_df["View Duration (minutes)"].apply(lambda x: 0 if x == '< 1'  else int(x))
recording_df['View Duration (minutes)'] = pd.to_numeric(
    recording_df['View Duration (minutes)'].replace('< 1', 0),
    errors='coerce'
).fillna(0).astype(int)


# Sum up the student viewtimes for live and recording respectively
live_df = live_df.groupby(["Email"], as_index=False)["Total duration (minutes)"].sum()
recording_df = recording_df.groupby(["Email"], as_index=False)["View Duration (minutes)"].sum()

# Concat both live and recording dataframes together
frames = [live_df, recording_df]
live_and_recording_df = pd.concat(frames)

# Group students by their email and sum up their
live_and_recording_df = live_and_recording_df.groupby(['Email']).agg({'Total duration (minutes)': 'sum', 'View Duration (minutes)': 'sum'})

# Reset email from being the index of the dataframe
live_and_recording_df = live_and_recording_df.reset_index()


live_and_recording_df["Total View Time (minutes)"] = live_and_recording_df["Total duration (minutes)"] + live_and_recording_df["View Duration (minutes)"]
live_and_recording_df.to_csv("./Debugging_csv's/live_and_recording.csv")
live_and_recording_df



    Date(UTC)            Name                   Email  View Duration (minutes)
0  03-02-2026    Seeta Genova     sgenova2024@fau.edu                    169.0
1  03-02-2026          Tom Le       tomle2023@fau.edu                    167.0
2  03-02-2026   Joseph Ortega     jortega2024@fau.edu                    169.0
3  03-02-2026  Jose Olascoaga  jolascoaga2022@fau.edu                     18.0
4  03-02-2026      Laiba Baig       lbaig2022@fau.edu                     27.0


,Email,Total duration (minutes),View Duration (minutes),Total View Time (minutes)
0,aallen2022@fau.edu,0.0,169.0,169.0
1,aaytac2023@fau.edu,0.0,138.0,138.0
2,abuleupesant2020@fau.edu,170.0,23.0,193.0
3,adaniel2023@fau.edu,169.0,0.0,169.0
4,adekermanjia2023@fau.edu,0.0,170.0,170.0
...,...,...,...,...
163,wsi2023@fau.edu,170.0,0.0,170.0
164,wtheodore2020@fau.edu,170.0,0.0,170.0
165,xnguyen2023@fau.edu,0.0,169.0,169.0
166,yjang2022@fau.edu,170.0,6.0,176.0


In [11]:
# Changing Total view time column to int dtype
int_dict = {'Total View Time (minutes)': int}
live_and_recording_df = live_and_recording_df.astype(int_dict)
print(live_and_recording_df.dtypes)

Email                         object
Total duration (minutes)     float64
View Duration (minutes)      float64
Total View Time (minutes)      int32
dtype: object


In [12]:
# Implementing grading scale function

def grading_scale(session_length, df):

  ninety_percent_watch = round(session_length * 0.9)
  eighty_percent_watch = round(session_length * 0.8)
  sixty_percent_watch = round(session_length * 0.6)
  forty_percent_watch = round(session_length * 0.4)
  thirty_percent_watch = round(session_length * 0.3)

  df['Grade'] = 'NA'
  df.loc[(df['Total View Time (minutes)'] < thirty_percent_watch), 'Grade'] = 0

  df.loc[(df['Total View Time (minutes)'] >= thirty_percent_watch) & (df['Total View Time (minutes)'] < forty_percent_watch), 'Grade'] = 1

  df.loc[(df['Total View Time (minutes)'] >= forty_percent_watch) & (df['Total View Time (minutes)'] < sixty_percent_watch), 'Grade'] = 2

  df.loc[(df['Total View Time (minutes)'] >= sixty_percent_watch) & (df['Total View Time (minutes)'] < eighty_percent_watch), 'Grade'] = 3

  df.loc[(df['Total View Time (minutes)'] >= eighty_percent_watch) & (df['Total View Time (minutes)'] < ninety_percent_watch), 'Grade'] = 4

  df.loc[(df['Total View Time (minutes)'] >= ninety_percent_watch), 'Grade'] = 5

In [13]:
# Call grading scale with live student attendance, recording student attendance, and session length csv

grading_scale(session_length, live_and_recording_df)

# Used in testing whether the grades were accurately applied to students in the gradebook
live_and_recording_df.to_csv("./Debugging_csv's/Preliminary.csv")

In [14]:
# Read in the course gradebook
course_gradebook_df = pd.read_csv(gradebook_name)

# Change zoom recording column name from email to SIS Login ID for joining with gradebook csv
live_and_recording_df = live_and_recording_df.rename(columns={"Email": "SIS Login ID"})

# ------------------------------------------------------------------
# Normalize emails ONLY for matching (do NOT modify Canvas identifiers)
# ------------------------------------------------------------------
course_gradebook_df["SIS Login ID"] = course_gradebook_df["SIS Login ID"].astype(str).str.strip()
live_and_recording_df["SIS Login ID"] = live_and_recording_df["SIS Login ID"].astype(str).str.strip()

# Create helper join key
course_gradebook_df["SIS_Login_norm"] = course_gradebook_df["SIS Login ID"].str.lower()

live_and_recording_df["SIS_Login_norm"] = (
    live_and_recording_df["SIS Login ID"]
      .str.lower()
      .str.replace("@health.fau.edu", "@fau.edu", regex=False)
)

# Copy for final grading
live_and_recording_df_final = live_and_recording_df.copy()

# ------------------------------------------------------------------
# Merge using normalized key (safe & predictable)
# ------------------------------------------------------------------
merged_df_experimental = pd.merge(
    course_gradebook_df,
    live_and_recording_df,
    on="SIS_Login_norm",
    how="left",
    suffixes=("_gradebook", "_zoom")
)

merged_df_final = pd.merge(
    course_gradebook_df,
    live_and_recording_df_final,
    on="SIS_Login_norm",
    how="left",
    suffixes=("_gradebook", "_zoom")
)

# ------------------------------------------------------------------
# Debugging CSV (to verify watch times & grades)
# ------------------------------------------------------------------
grade_verification_df = pd.concat([
    merged_df_experimental[["Student", assignment_name]],
    merged_df_experimental[[
        "SIS Login ID_gradebook",
        "Total duration (minutes)",
        "View Duration (minutes)",
        "Total View Time (minutes)",
        "Grade"
    ]]
], axis=1)

grade_verification_df.to_csv("./Debugging_csv's/grade_verification.csv", index=False)

# ------------------------------------------------------------------
# Move attendance grades into assignment column
# ------------------------------------------------------------------
merged_df_experimental.loc[2:, assignment_name] = merged_df_experimental.loc[2:, "Grade"]
merged_df_final.loc[2:, assignment_name] = merged_df_final.loc[2:, "Grade"]

# ------------------------------------------------------------------
# Drop unwanted columns from final dataframe
# ------------------------------------------------------------------
merged_df_final = merged_df_final.drop(
    ["Total duration (minutes)", "View Duration (minutes)", "Total View Time (minutes)", "Grade"],
    axis=1
)

# ------------------------------------------------------------------
# Fill missing values (no attendance → 0)
# ------------------------------------------------------------------
merged_df_experimental = merged_df_experimental.fillna(value={assignment_name: 0})
merged_df_experimental.loc[2:] = merged_df_experimental.loc[2:].fillna(value={"Grade": 0})
merged_df_final = merged_df_final.fillna(value={assignment_name: 0})

# ------------------------------------------------------------------
# Export debugging CSV
# ------------------------------------------------------------------
merged_df_experimental.to_csv("./Debugging_csv's/Testing.csv", index=False)

# ------------------------------------------------------------------
# Prepare final Canvas-ready CSV
# ------------------------------------------------------------------
# Keep ONLY gradebook SIS Login ID and rename it back
merged_df_final = merged_df_final.rename(
    columns={"SIS Login ID_gradebook": "SIS Login ID"}
)

# Drop zoom SIS Login ID and helper join key
merged_df_final = merged_df_final.drop(
    columns=["SIS Login ID_zoom", "SIS_Login_norm"],
    errors="ignore"
)

# Final gradebook to submit to Canvas
merged_df_final.to_csv("grading_results.csv", index=False)


C:\Users\pakal\AppData\Local\Temp\ipykernel_20508\2382279171.py:78: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged_df_experimental.loc[2:] = merged_df_experimental.loc[2:].fillna(value={"Grade": 0})


In [15]:
# Gives grade counts for the class (Not yet complete)
grade_count = merged_df_experimental.copy()
grade_count = grade_count.groupby(['Grade'])['Grade'].size()
grade_count.to_csv('grade_count.csv')

In [16]:
# Used to compare the gradebook df with the merged_df_final to see if it only changed the assignment name column in the gradebook

column_list = list(course_gradebook_df.columns)
column_list.remove(assignment_name)

# NEW: remove helper column(s) that are not meant to exist in final output
if "SIS_Login_norm" in column_list:
    column_list.remove("SIS_Login_norm")

# If you used suffix merges earlier, these may appear in merged_df_final / gradebook, remove if present
if "SIS Login ID_zoom" in column_list:
    column_list.remove("SIS Login ID_zoom")
if "SIS Login ID_gradebook" in column_list:
    column_list.remove("SIS Login ID_gradebook")

comparison_df = merged_df_final.merge(
    course_gradebook_df,
    on=column_list,
    how="outer",
    suffixes=["", "_"],
    indicator=True
)

comparison_df.to_csv("./Debugging_csv's/comparison.csv", index=False)
